# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List all record sets and their fields referencing them by @id
record_sets = metadata.record_sets
print(f"Available record sets (@id):\n")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")
    fields = rs.get('fields', [])
    print("  Fields in this record set:")
    for field in fields:
        print(f"     - {field['@id']}: {field.get('name', field['@id'])} | DataType: {field.get('dataType', 'N/A')}")
    columns = rs.get('columns', [])
    if columns:
        print("  Columns in this record set:")
        for column in columns:
            print(f"     - {column['@id']}: {column.get('name', column['@id'])} | Path: {column.get('path', 'N/A')}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, using the record set and field `@id`s from the overview.

In [ ]:
# List all record_set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}
print(f"Attempting to extract data from the following record sets: {record_set_ids}\n")

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set: {record_set_id}")
        else:
            print(f"No records found for record set: {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# For demonstration, display column names of the first populated record set
main_record_set_id = None
for rs_id, df in dataframes.items():
    if len(df.columns) > 0:
        main_record_set_id = rs_id
        print(f"First populated record set: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
        break
if main_record_set_id is None:
    print("No populated record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
df = dataframes.get(main_record_set_id, pd.DataFrame())
if df.empty or len(df.columns) == 0:
    print("No data available for EDA.")
else:
    # Display column types
    print("Column data types:")
    print(df.dtypes)

    # Attempt to infer a numeric field for demonstration
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try to coerce columns to numeric
        for col in df.columns:
            try:
                df[col+'_num'] = pd.to_numeric(df[col], errors='coerce')
                if df[col+'_num'].notna().sum() > 0:
                    numeric_candidates.append(col+'_num')
            except:
                continue
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        threshold = df[numeric_field_id].quantile(0.75)  # 75th percentile for filter demo
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by a likely categorical field
        group_field = None
        # Prefer 'Group', 'Sample', 'Type' etc.
        for candidate in ['Group', 'Sample', 'Type', 'group', 'sample', 'type']:
            if candidate in filtered_df.columns:
                group_field = candidate
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and numeric_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f'Distribution of {numeric_field_id} (filtered)')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for plotting.")

## 6. Conclusion
This notebook showed how to use the `mlcroissant` library to load, explore, and perform basic EDA and visualization on a dataset defined by a Croissant schema. You can use this framework to guide deeper analysis, feature engineering, or integration with machine learning workflows.